# Raman Consensus Spectra — NMF & Wavelet Decomposition with Interpretability Analysis

**Input:** `raman_consensus_spectra.xlsx` — five consensus curves (Arithmetic Mean, B-spline Mean, Robust Median Centre, Geometric Median, Wasserstein Barycenter) over 2118 wavenumber points (400–3500 cm⁻¹).

**Pipeline:**
1. Non-negative Matrix Factorisation (NMF) — latent spectral components shared across consensus methods
2. Continuous Wavelet Transform (CWT) — multi-scale time-frequency decomposition per consensus spectrum
3. Discrete Wavelet Transform (DWT) — multi-resolution approximation/detail decomposition
4. Interpretability analysis — mapping every decomposed component to known Raman markers of milk

> **Data availability:** This notebook reads the unaligned `raman_consensus_spectra.xlsx` (400–3500 cm⁻¹), which is not included in this repository. The consensus spectra are provided on the common 402–3499 cm⁻¹ grid as `data/raman_consensus_spectra_aligned_402_3499.xlsx`. This notebook is included to document the processing steps; its file paths refer to the original project layout.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import FancyArrowPatch
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler
import pywt
from scipy import signal

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

# ── Palette for the five consensus methods ────────────────────────────────────
CONSENSUS_COLORS = {
    'Arithmetic Mean':        'crimson',
    'B-spline Mean':          'darkorange',
    'Robust Median Centre':   'green',
    'Geometric Median':       'darkgreen',
    'Wasserstein Barycenter': 'purple',
}

## 1. Load Consensus Spectra

In [ ]:
df = pd.read_excel('raman_consensus_spectra.xlsx', index_col=0)
x  = df.index.values.astype(float)   # wavenumber axis
methods = list(df.columns)

print(f'Wavenumber range : {x.min():.1f} – {x.max():.1f} cm⁻¹')
print(f'Data points      : {len(x)}')
print(f'Consensus methods: {methods}')
df.head(3)

## 2. Known Raman Markers of Milk

Reference table used throughout for all interpretability mapping.

In [ ]:
MARKERS = pd.DataFrame([
    (605.2,  'Lipid',   'C=O in-plane deformation / glycerol C–O–C skeletal bending (triglycerides)'),
    (874.8,  'Lipid',   'C–C stretching of –(CH₂)ₙ– chain; tryptophan / monosaccharide C–O–C contributions'),
    (1001.5, 'Protein', 'Phenylalanine symmetric ring breathing — unambiguous protein marker (casein/whey)'),
    (1062.6, 'Lipid',   'Skeletal C–C stretch of all-trans fatty acid chain; sensitive to chain order'),
    (1079.0, 'Lipid',   'C–C stretching of –(CH₂)ₙ– fatty acid chain; lactose C–O vibrations overlap'),
    (1123.1, 'Lipid',   'C–C / C–O–C stretching in glycerol ester backbone of triglycerides'),
    (1155.7, 'Lactose', 'C–O bond stretching of polysaccharides — primary lactose marker in milk'),
    (1265.5, 'Lipid',   'In-plane δ(=C–H) deformation of cis double bonds (unsaturated FA); Amide III overlap'),
    (1300.7, 'Lipid',   'In-phase CH₂ twisting (τCH₂) — saturated fatty acid chains'),
    (1438.5, 'Lipid',   'CH₂ scissoring / δ(CH₂) deformation — dominant saturated fatty acid methylene band'),
    (1521.4, 'Lipid',   'Carotenoid C=C conjugated stretch (β-carotene)'),
    (1652.5, 'Protein', 'Amide I (protein C=O stretch, α-helix) overlapping with C=C of unsaturated FA'),
    (1743.8, 'Lipid',   'ν(C=O) ester carbonyl stretch — triglycerides and free fatty acids'),
    (2851.7, 'Lipid',   'Symmetric CH₂ stretching (νs CH₂) — fatty acid alkyl chains'),
    (2891.2, 'Lipid',   'Asymmetric CH₂ stretching (νas CH₂) — fatty acid chains'),
    (2929.1, 'Lipid',   'Asymmetric CH₃ stretching (νas CH₃) / Fermi resonance component of CH₂ stretch'),
    (3171.8, 'Water',   'O–H stretching — low-frequency shoulder of the broad water band (H-bonded O–H)'),
], columns=['Wavenumber', 'Category', 'Assignment'])

CAT_COLOR = {'Lipid': '#e08030', 'Protein': '#3070b0', 'Lactose': '#40a060', 'Water': '#9060c0'}
print(MARKERS[['Wavenumber','Category','Assignment']].to_string(index=False))

[1] Surkova, A. & Bogomolov, A. (2023)

Analysis of Milk Microstructure Using Raman Hyperspectral Imaging

Molecules, 28(6), 2770.

DOI: 10.3390/molecules28062770

— Supports: lactose C–O at 1144 cm⁻¹, CH₂ twisting at 1309 cm⁻¹, symmetric CH₂ stretch at 2865 cm⁻¹, Amide I at ~1670 cm⁻¹ with fat overlap
[2] de Wolf, J.R. et al. (2021)

Evaluation of the changes in human milk lipid composition and conformational state with Raman spectroscopy during a breastfeed

Biomedical Optics Express, 12(7), 3934–3947.

DOI: 10.1364/BOE.427646

— Supports: ester C=O at 1745 cm⁻¹, CH₂ scissoring at 1440 cm⁻¹, unsaturation markers at 1260 and 1655 cm⁻¹, saturated FA marker at 1300 cm⁻¹
[3] Czamara, K., Majzner, K., Pacia, M.Z., Kochan, K., Kaczor, A. & Baranska, M. (2015)

Raman spectroscopy of lipids: a review

Journal of Raman Spectroscopy, 46(1), 4–20.

DOI: 10.1002/jrs.4607

— Supports: all major lipid band assignments — CH₂ symmetric stretch at 2851 cm⁻¹, CH₂ asymmetric stretch at 2881 cm⁻¹, CH₂ scissoring at 1440 cm⁻¹, C=O ester at ~1740 cm⁻¹, C–C chain stretches 1060–1130 cm⁻¹
[4] Greig, J.C., Tipping, W.J., Graham, D., Faulds, K. & Gould, G.W. (2024)

New insights into lipid and fatty acid metabolism from Raman spectroscopy

Analyst, 149(19), 4789–4810.

DOI: 10.1039/D4AN00846D

— Supports: CH₂ symmetric stretch at 2851 cm⁻¹, CH₂ asymmetric stretch at 2881 cm⁻¹, scissoring at 1440 cm⁻¹, ester CO at 1740 cm⁻¹, unsaturation at 1262 and 1655 cm⁻¹
[5] Rygula, A., Majzner, K., Marzec, K.M., Kaczor, A., Pilarczyk, M. & Baranska, M. (2013)

Raman spectroscopy of proteins: a review

Journal of Raman Spectroscopy, 44(8), 1061–1076.

DOI: 10.1002/jrs.4335

— Supports: phenylalanine at 1001–1003 cm⁻¹, Amide I band position and α-helix assignment, tryptophan at 876 cm⁻¹
[6] Acharya, B.S., Nair, S. & Abdul Salam, A.A. (2026)

Quality Analysis and Detection of Adulterants and Contaminations in Milk/Milk Powder by Raman Spectroscopy

Comprehensive Reviews in Food Science and Food Safety.

DOI: 10.1111/1541-4337.70403

— Supports: overall milk Raman marker framework, adulteration detection context, SERS and hyperspectral Raman in dairy
[7] Petrov, D. et al. / carotenoid resonance Raman literature (2008)

Resonance Raman measurements of carotenoids using light emitting diodes

arXiv: 0801.1697

— Supports: β-carotene at 1008 cm⁻¹ (C–CH₃ rocking), 1159 cm⁻¹ (C–C stretch), 1525 cm⁻¹ (C=C stretch, strongest mode)
[8] Lipid fat globule confocal Raman (2015)

Milk fat globules by confocal Raman microscopy: Differences in human, bovine and caprine milk

Food Research International, 80, 136–144.

DOI: 10.1016/j.foodres.2015.12.034

— Supports: carotenoid detection in bovine milk fat globules, 2885/2850 cm⁻¹ ratio as fat globule state marker
[9] Urbanová, M. et al. (WSEAS, 2016)

Raman spectroscopy as a modern tool for lactose determination

WSEAS Transactions on Biology and Biomedicine, 13, 77–84.

— Supports: lactose quantification at 1087 cm⁻¹, phenylalanine at 1001 cm⁻¹ as internal reference standard in milk, C–O–H bending mode for lactose
[10] Bērziņš, K. et al. (2021)

Qualitative and quantitative vibrational spectroscopic analysis of macronutrients in breast milk

Spectrochimica Acta Part A, 246, 118982.

DOI: 10.1016/j.saa.2020.118982

— Supports: comprehensive milk macronutrient Raman band assignments across fat, protein, and lactose regions

---
## 3. Non-negative Matrix Factorisation (NMF)

The consensus matrix **X** (2118 wavenumbers × 5 methods) is factorised as **X ≈ W H**, where:
- **W** (2118 × r) — *spectral basis components*: latent pure-component spectra
- **H** (r × 5) — *activations*: how much each consensus method draws on each component

NMF is well-suited here because Raman intensities are non-negative and the non-negativity constraint encourages parts-based, physically interpretable components.

### 3.1 Choose Number of Components — Reconstruction Error & Explained Variance

In [ ]:
# X: (n_wavenumbers, n_methods) — each column is a consensus spectrum
X = df.values.copy()                   # (2118, 5)
X = np.clip(X, 0, None)               # ensure non-negative (Wasserstein may have tiny negatives after interp)

total_var = np.sum(X ** 2)
r_range   = range(1, 6)
errors, explained = [], []

for r in r_range:
    model = NMF(n_components=r, init='nndsvda', max_iter=2000, random_state=42)
    W = model.fit_transform(X)
    H = model.components_
    recon_err = np.sum((X - W @ H) ** 2)
    errors.append(recon_err)
    explained.append(1 - recon_err / total_var)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(r_range), errors, 'o-', color='steelblue')
axes[0].set_xlabel('Number of components (r)')
axes[0].set_ylabel('Reconstruction error (Frobenius²)')
axes[0].set_title('NMF Reconstruction Error vs r')
axes[0].set_xticks(list(r_range))

axes[1].bar(list(r_range), [v * 100 for v in explained], color='steelblue', alpha=0.8)
axes[1].set_xlabel('Number of components (r)')
axes[1].set_ylabel('Explained variance (%)')
axes[1].set_title('NMF Explained Variance vs r')
axes[1].set_xticks(list(r_range))

plt.tight_layout()
plt.show()

for r, e, v in zip(r_range, errors, explained):
    print(f'r={r}: reconstruction error = {e:.4e}, explained variance = {v*100:.2f}%')

### 3.2 Fit NMF with r = 3 Components

In [ ]:
R = 3
nmf = NMF(n_components=R, init='nndsvda', max_iter=2000, random_state=42)
W   = nmf.fit_transform(X)     # (2118, 3) — spectral basis
H   = nmf.components_          # (3, 5)    — activations per consensus method
X_recon = W @ H

recon_err = np.sum((X - X_recon) ** 2)
exp_var   = 1 - recon_err / total_var
print(f'NMF r={R}: reconstruction error = {recon_err:.4e}, explained variance = {exp_var*100:.2f}%')

# L2-normalise each component for plotting (scale captured in H)
norms   = np.linalg.norm(W, axis=0, keepdims=True)
W_norm  = W / norms
H_scaled = H * norms.T

### 3.3 NMF Spectral Components with Marker Annotations

In [ ]:
COMP_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c']
TOL = 12  # ±12 cm⁻¹ window for peak-to-marker matching

fig, axes = plt.subplots(R, 1, figsize=(14, 4 * R), sharex=True)

for i, ax in enumerate(axes):
    comp = W_norm[:, i]
    ax.plot(x, comp, color=COMP_COLORS[i], linewidth=1.4, label=f'NMF Component {i+1}')
    ax.fill_between(x, 0, comp, color=COMP_COLORS[i], alpha=0.15)

    # Detect prominent peaks in this component
    peaks, props = signal.find_peaks(comp, prominence=0.01 * comp.max(), distance=8)

    for pk in peaks:
        wn = x[pk]
        # Match to nearest known marker within tolerance
        dists = np.abs(MARKERS['Wavenumber'] - wn)
        idx   = dists.idxmin()
        if dists[idx] <= TOL:
            row   = MARKERS.loc[idx]
            color = CAT_COLOR[row['Category']]
            ax.axvline(wn, color=color, linewidth=0.8, linestyle='--', alpha=0.7)
            ax.annotate(
                f'{wn:.0f}\n[{row["Category"]}]',
                xy=(wn, comp[pk]),
                xytext=(0, 8), textcoords='offset points',
                fontsize=7, ha='center', color=color,
                rotation=90, va='bottom'
            )
        else:
            ax.axvline(wn, color='grey', linewidth=0.6, linestyle=':', alpha=0.5)
            ax.annotate(
                f'{wn:.0f}', xy=(wn, comp[pk]),
                xytext=(0, 5), textcoords='offset points',
                fontsize=6, ha='center', color='grey', rotation=90
            )

    ax.set_ylabel('Loading (a.u.)')
    ax.set_title(f'NMF Component {i+1}')
    ax.legend(loc='upper right', fontsize=9)

    # Category legend patches
    from matplotlib.patches import Patch
    handles = [Patch(color=c, label=cat) for cat, c in CAT_COLOR.items()]
    ax.legend(handles=handles, fontsize=7, loc='upper left', title='Marker category', ncol=4)

axes[-1].set_xlabel('Raman Shift (cm⁻¹)')
plt.suptitle(f'NMF Spectral Components (r={R}) — Annotated with Milk Raman Markers', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 3.4 NMF Activation Matrix — How Each Consensus Method Draws on Each Component

In [ ]:
# Normalise H rows so each component's activation sums to 1 across methods (fractional contribution)
H_frac = H_scaled / H_scaled.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw activation heatmap
ax = axes[0]
im = ax.imshow(H_scaled, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(methods)))
ax.set_xticklabels([m.replace(' ', '\n') for m in methods], fontsize=8)
ax.set_yticks(range(R))
ax.set_yticklabels([f'Component {i+1}' for i in range(R)])
ax.set_title('NMF Activation Matrix H (raw)')
plt.colorbar(im, ax=ax, label='Activation')
for i in range(R):
    for j in range(len(methods)):
        ax.text(j, i, f'{H_scaled[i,j]:.3f}', ha='center', va='center', fontsize=7)

# Right: stacked bar — component mix per consensus method
ax = axes[1]
bottom = np.zeros(len(methods))
for i in range(R):
    ax.bar(range(len(methods)), H_frac[i], bottom=bottom,
           color=COMP_COLORS[i], label=f'Component {i+1}', alpha=0.85)
    bottom += H_frac[i]
ax.set_xticks(range(len(methods)))
ax.set_xticklabels([m.replace(' ', '\n') for m in methods], fontsize=8)
ax.set_ylabel('Fractional activation')
ax.set_title('Component Mix per Consensus Method')
ax.legend(fontsize=9)

plt.suptitle('NMF Activations — Component Contribution per Consensus Method', fontsize=12)
plt.tight_layout()
plt.show()

### 3.5 NMF Reconstruction Quality per Consensus Method

In [ ]:
fig, axes = plt.subplots(len(methods), 1, figsize=(14, 3 * len(methods)), sharex=True)

for ax, method in zip(axes, methods):
    j = methods.index(method)
    orig  = X[:, j]
    recon = X_recon[:, j]
    residual = orig - recon
    r2 = 1 - np.sum(residual**2) / np.sum((orig - orig.mean())**2)

    ax.plot(x, orig,     color=CONSENSUS_COLORS[method], linewidth=1.2, label='Original')
    ax.plot(x, recon,    color='black', linewidth=1.0, linestyle='--', alpha=0.8, label='NMF reconstruction')
    ax.fill_between(x, orig, recon, color='red', alpha=0.08, label='Residual')
    ax.set_ylabel('Intensity (a.u.)', fontsize=8)
    ax.set_title(f'{method}  (R² = {r2:.5f})', fontsize=9)
    ax.legend(fontsize=7, loc='upper right')

axes[-1].set_xlabel('Raman Shift (cm⁻¹)')
plt.suptitle(f'NMF Reconstruction vs Original — r={R}', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

### 3.6 NMF Component-to-Marker Interpretability Table

In [ ]:
rows = []
for i in range(R):
    comp  = W_norm[:, i]
    peaks, _ = signal.find_peaks(comp, prominence=0.01 * comp.max(), distance=8)
    for pk in peaks:
        wn   = x[pk]
        dists = np.abs(MARKERS['Wavenumber'] - wn)
        idx   = dists.idxmin()
        match = MARKERS.loc[idx]
        offset = dists[idx]
        rows.append({
            'Component':      f'NMF-{i+1}',
            'Peak (cm⁻¹)':   round(wn, 1),
            'Loading':        round(comp[pk], 5),
            'Nearest marker': round(match['Wavenumber'], 1),
            'Offset (cm⁻¹)': round(offset, 1),
            'Category':       match['Category'] if offset <= TOL else '—',
            'Assignment':     match['Assignment'] if offset <= TOL else 'Unassigned',
        })

interp_df = pd.DataFrame(rows)
interp_df = interp_df.sort_values(['Component', 'Loading'], ascending=[True, False])
print(interp_df.to_string(index=False))

---
## 4. Wavelet Transform Decomposition

Wavelets decompose each consensus spectrum into contributions at different *scales* (spatial frequencies), separating broad baseline features, mid-scale spectral bands, and sharp fine-structure peaks.

- **Continuous Wavelet Transform (CWT)** — scalogram showing spectral energy as a function of both wavenumber and scale
- **Discrete Wavelet Transform (DWT) multi-resolution analysis** — hierarchical approximation and detail levels

### 4.1 Wavelet Scale–Wavenumber Correspondence

The DWT is performed with the **Daubechies-4 (db4)** wavelet. Level-1 detail captures the finest features (~2 point spacing ≈ 3.5 cm⁻¹), level-2 captures ~7 cm⁻¹, and so on, doubling at each level. We use 6 levels.

In [ ]:
WAVELET  = 'db4'
N_LEVELS = 6
dx       = np.median(np.diff(x))   # wavenumber spacing ≈ 1.77 cm⁻¹

print(f'Wavenumber spacing: {dx:.2f} cm⁻¹')
print(f'Wavelet            : {WAVELET}')
print(f'Decomposition levels: {N_LEVELS}')
print()
print('Level | Approx feature width (cm⁻¹) | Physical interpretation')
print('------+-----------------------------+---------------------------------------------')
for lvl in range(1, N_LEVELS + 1):
    width = dx * (2 ** lvl)
    if width < 10:
        interp = 'Sub-band fine structure / noise'
    elif width < 30:
        interp = 'Sharp Raman peaks (protein, lactose ring modes)'
    elif width < 80:
        interp = 'Moderate-width peaks (lipid C–C, C=O bands)'
    elif width < 200:
        interp = 'Broad peaks (CH₂/CH₃ stretch cluster, Amide I)'
    elif width < 500:
        interp = 'Very broad features (water O–H envelope)'
    else:
        interp = 'Baseline / slow spectral drift'
    print(f'  D{lvl}  |  {width:>7.1f}                    | {interp}')

### 4.2 DWT Multi-Resolution Decomposition — All Consensus Methods

In [ ]:
def dwt_reconstruct_levels(y, wavelet, levels):
    """Return reconstructed detail signals D1..Dn and approximation An, all on the original grid."""
    coeffs = pywt.wavedec(y, wavelet, level=levels)
    # coeffs = [cA_n, cD_n, cD_{n-1}, ..., cD_1]
    details = []
    for k in range(1, levels + 1):  # k=1 → finest (D1), k=levels → coarsest detail (Dn)
        zero_coeffs = [np.zeros_like(c) for c in coeffs]
        zero_coeffs[levels + 1 - k] = coeffs[levels + 1 - k]
        rec = pywt.waverec(zero_coeffs, wavelet)
        details.append(rec[:len(y)])
    # Approximation (low-freq baseline)
    zero_coeffs = [np.zeros_like(c) for c in coeffs]
    zero_coeffs[0] = coeffs[0]
    approx = pywt.waverec(zero_coeffs, wavelet)[:len(y)]
    return details, approx  # details[0]=D1 (fine), details[-1]=Dn (coarse)

# Compute for all methods
dwt_results = {}
for method in methods:
    j = methods.index(method)
    y = X[:, j]
    details, approx = dwt_reconstruct_levels(y, WAVELET, N_LEVELS)
    dwt_results[method] = {'details': details, 'approx': approx}

print('DWT decomposition complete for all consensus methods.')

In [ ]:
# Plot DWT detail levels for one representative method (Arithmetic Mean)
method_plot = 'Arithmetic Mean'
details = dwt_results[method_plot]['details']
approx  = dwt_results[method_plot]['approx']

level_labels = [f'D{k+1}  (~{dx*(2**(k+1)):.0f} cm⁻¹)' for k in range(N_LEVELS)]
level_labels.append(f'A{N_LEVELS} (baseline)')
all_signals = details + [approx]

fig, axes = plt.subplots(N_LEVELS + 2, 1, figsize=(14, 3 * (N_LEVELS + 2)), sharex=True)

# Original
axes[0].plot(x, X[:, methods.index(method_plot)], color=CONSENSUS_COLORS[method_plot], linewidth=1.2)
axes[0].set_title(f'Original — {method_plot}')
axes[0].set_ylabel('Intensity')

for ax, sig, label in zip(axes[1:], all_signals, level_labels):
    ax.plot(x, sig, linewidth=0.9, color='steelblue')
    ax.axhline(0, color='grey', linewidth=0.4, linestyle='--')
    ax.set_title(label)
    ax.set_ylabel('Amplitude')

    # Annotate known markers on detail levels (not on approximation)
    if 'A' not in label:
        for _, row in MARKERS.iterrows():
            if x.min() <= row['Wavenumber'] <= x.max():
                ax.axvline(row['Wavenumber'], color=CAT_COLOR[row['Category']],
                           linewidth=0.6, linestyle=':', alpha=0.6)

axes[-1].set_xlabel('Raman Shift (cm⁻¹)')
plt.suptitle(f'DWT Multi-Resolution Decomposition ({WAVELET}, {N_LEVELS} levels) — {method_plot}',
             fontsize=12, y=1.005)
plt.tight_layout()
plt.show()

### 4.3 DWT Detail Energy per Level — Which Scales Carry the Most Spectral Information?

In [ ]:
level_names = [f'D{k+1}' for k in range(N_LEVELS)] + [f'A{N_LEVELS}']
energy_df   = pd.DataFrame(index=methods, columns=level_names, dtype=float)

for method in methods:
    res    = dwt_results[method]
    total  = np.sum(X[:, methods.index(method)] ** 2)
    for k, det in enumerate(res['details']):
        energy_df.loc[method, f'D{k+1}'] = np.sum(det**2) / total
    energy_df.loc[method, f'A{N_LEVELS}'] = np.sum(res['approx']**2) / total

fig, ax = plt.subplots(figsize=(12, 5))
bar_width = 0.15
xpos = np.arange(len(level_names))
for i, method in enumerate(methods):
    ax.bar(xpos + i * bar_width, energy_df.loc[method].values * 100,
           width=bar_width, label=method, color=list(CONSENSUS_COLORS.values())[i], alpha=0.85)

ax.set_xticks(xpos + bar_width * (len(methods) - 1) / 2)
ax.set_xticklabels(
    [f'{n}\n(~{dx*(2**(k+1)):.0f} cm⁻¹)' for k, n in enumerate(level_names[:-1])] + [f'A{N_LEVELS}\n(baseline)'],
    fontsize=8
)
ax.set_ylabel('Relative energy (%)')
ax.set_title('DWT Detail Energy per Level — All Consensus Methods')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

print(energy_df.round(4).to_string())

### 4.4 Continuous Wavelet Transform (CWT) Scalograms

In [ ]:
# CWT with Morlet wavelet; scales chosen to cover ~2 to ~400 cm⁻¹ feature widths
scales = np.logspace(0, np.log10(len(x) / 8), 120).astype(int)
scales = np.unique(scales)
# Convert scale index → approximate cm⁻¹ width  (scale * dx * central freq factor for morl)
# pywt Morlet central frequency ≈ 0.8125 cycles/sample
cwt_wavelet = 'morl'
fc = pywt.central_frequency(cwt_wavelet)
scale_cm = fc * dx / (scales / len(x) * dx + 1e-12)  # rough conversion
# Simpler: use scales directly as x-axis (log scale) and annotate

fig, axes = plt.subplots(len(methods), 1, figsize=(14, 4 * len(methods)))

for ax, method in zip(axes, methods):
    j   = methods.index(method)
    y   = X[:, j]
    cfs, freqs = pywt.cwt(y, scales, cwt_wavelet, sampling_period=dx)
    power = np.abs(cfs) ** 2

    im = ax.contourf(x, scales, power, levels=50, cmap='inferno')
    ax.set_yscale('log')
    ax.set_ylabel('Scale (log)')
    ax.set_title(f'CWT Scalogram — {method}')

    # Mark known marker wavenumbers
    for _, row in MARKERS.iterrows():
        if x.min() <= row['Wavenumber'] <= x.max():
            ax.axvline(row['Wavenumber'], color=CAT_COLOR[row['Category']],
                       linewidth=0.8, linestyle='--', alpha=0.8)

    plt.colorbar(im, ax=ax, label='Power')

axes[-1].set_xlabel('Raman Shift (cm⁻¹)')
plt.suptitle('CWT Scalograms (Morlet) — All Consensus Methods\n'
             '(dashed lines = known milk Raman markers)', fontsize=12, y=1.005)
plt.tight_layout()
plt.show()

---
## 5. Integrated Interpretability Analysis

Consolidates findings from NMF and wavelet decomposition into a per-marker attribution table and a summary heatmap.

### 5.1 Per-Marker Analysis: NMF Component Loading + DWT Level Energy at Each Marker

In [ ]:
# For each known marker: find the nearest wavenumber index, then record
# (a) which NMF component has the highest loading at that position
# (b) which DWT level has the highest energy in a ±20 cm⁻¹ window around the marker

marker_rows = []
for _, mrow in MARKERS.iterrows():
    wn  = mrow['Wavenumber']
    idx = np.argmin(np.abs(x - wn))  # nearest grid index
    window = (x >= wn - 20) & (x <= wn + 20)

    # NMF: loading of each component at this wavenumber
    comp_loadings = W_norm[idx, :]  # shape (R,)
    dominant_comp = np.argmax(comp_loadings) + 1
    dominant_load = comp_loadings.max()

    # DWT: energy in window for the arithmetic mean (representative)
    j = methods.index('Arithmetic Mean')
    res = dwt_results['Arithmetic Mean']
    level_energies = [np.sum(det[window]**2) for det in res['details']]
    dominant_level = np.argmax(level_energies) + 1
    approx_width   = dx * (2 ** dominant_level)

    marker_rows.append({
        'Wavenumber':      wn,
        'Category':        mrow['Category'],
        'Assignment':      mrow['Assignment'][:60] + ('…' if len(mrow['Assignment']) > 60 else ''),
        'NMF component':   f'NMF-{dominant_comp}',
        'NMF loading':     round(dominant_load, 5),
        'DWT level':       f'D{dominant_level} (~{approx_width:.0f} cm⁻¹)',
    })

attr_df = pd.DataFrame(marker_rows)
print(attr_df.to_string(index=False))

### 5.3 Annotated Consensus Spectrum — Full Interpretability Summary

In [ ]:
# Grand mean of all consensus methods as the reference spectrum for annotation
grand = X.mean(axis=1)

fig, ax = plt.subplots(figsize=(16, 6))

# All consensus methods in background
for method in methods:
    j = methods.index(method)
    ax.plot(x, X[:, j], color=CONSENSUS_COLORS[method], linewidth=0.8, alpha=0.4)

# Grand mean on top
ax.plot(x, grand, color='black', linewidth=1.6, label='Grand mean (all consensus)')

# Annotate each known marker
for _, row in MARKERS.iterrows():
    wn    = row['Wavenumber']
    if not (x.min() <= wn <= x.max()):
        continue
    idx   = np.argmin(np.abs(x - wn))
    yval  = grand[idx]
    color = CAT_COLOR[row['Category']]

    # NMF dominant component at this position
    comp_loadings = W_norm[idx, :]
    dom_comp = np.argmax(comp_loadings) + 1

    ax.annotate(
        f"{wn:.0f}\n[{row['Category']}]\nNMF-{dom_comp}",
        xy=(wn, yval),
        xytext=(wn, yval + 0.015 + 0.005 * (list(MARKERS['Wavenumber']).index(wn) % 3)),
        fontsize=6, color=color, ha='center', va='bottom',
        arrowprops=dict(arrowstyle='->', color=color, lw=0.6),
    )

# Category legend
from matplotlib.patches import Patch
handles = ([plt.Line2D([0],[0], color='black', linewidth=1.5, label='Grand mean')] +
           [Patch(color=c, label=cat) for cat, c in CAT_COLOR.items()])
ax.legend(handles=handles, fontsize=8, loc='upper right')
ax.set_xlabel('Raman Shift (cm⁻¹)')
ax.set_ylabel('Normalised Intensity (a.u.)')
ax.set_title('Consensus Spectrum — Full Interpretability Map\n'
             '(marker category + dominant NMF component at each band)', fontsize=11)
plt.tight_layout()
plt.show()

### 5.4 Save Interpretability Table

In [ ]:
# Merge NMF component-to-marker table with DWT dominant level
attr_df['DWT dominant level'] = attr_df['DWT level']

# Add all three NMF component loadings
for i in range(R):
    attr_df[f'NMF-{i+1} loading'] = [
        round(W_norm[np.argmin(np.abs(x - r['Wavenumber'])), i], 5)
        for _, r in MARKERS.iterrows()
    ]

out_path = 'raman_interpretability.xlsx'
with pd.ExcelWriter(out_path) as writer:
    attr_df.to_excel(writer, sheet_name='Marker Attribution', index=False)
    interp_df.to_excel(writer, sheet_name='NMF Peak Assignments', index=False)
    energy_df.to_excel(writer, sheet_name='DWT Level Energies')

print(f'Saved → {out_path}')
attr_df

# Summary of Findings — NMF & Wavelet Decomposition

- This notebook applied NMF and Discrete/Continuous Wavelet Transform to the five consensus
  Raman spectra to decompose them into interpretable sub-components and validate their
  correspondence with known Raman markers of milk biochemistry.

- Three NMF components reconstructed all five consensus spectra with 99.99% explained
  variance, confirming that all five consensus methods — Arithmetic Mean, B-spline Mean,
  Robust Median Centre, Geometric Median, and Wasserstein Barycenter — produce highly
  consistent representations of the same underlying milk signal. The NMF components
  themselves partition the signal by peak intensity rather than biochemical class, and
  are therefore more useful as a validation of consensus consistency than as a tool for
  biochemical interpretation.

- The DWT energy distribution is the more informative output of this analysis. Across the
  consensus spectra, approximately 78% of spectral energy resides in the baseline (A6),
  13.5% in D6 (~93 cm⁻¹), and 5.7% in D5 (~46 cm⁻¹), with D1 and D2 carrying
  effectively zero energy — confirming the consensus spectra are clean and noise-free
  at sub-resolution scales.

- D6 captures the broad, dominant lipid peaks: CH₂ scissoring (1438.5 cm⁻¹) and the
  CH₂/CH₃ stretching cluster (2851–2929 cm⁻¹). D5 captures the moderate-width
  biochemically specific bands: Amide I (1652.5 cm⁻¹) representing protein content,
  and the lactose C–O stretch (1155.7 cm⁻¹). Together, D5 and D6 therefore encode the
  fat, protein, and lactose balance of genuine unadulterated milk.

- The Wasserstein Barycenter concentrates ~93% of its energy in A6 compared to ~78%
  for the other four methods, reflecting its stronger smoothing of sharp peak structure.
  This should be considered when selecting a consensus reference — the arithmetic and
  B-spline means preserve biochemically informative peak detail more faithfully.

- The CWT scalogram provides a visual complement to the DWT energy analysis. It is read
  as a 2D heatmap where the x-axis is wavenumber position and the y-axis is scale on a
  log axis — low scale at the bottom corresponds to fine, sharp features while high scale
  at the top corresponds to broad, slowly-varying features. The colour (power) at each
  point is the squared magnitude of the wavelet coefficient, representing how much signal
  energy exists at that wavenumber at that particular scale. A bright vertical stripe
  therefore indicates a real spectral feature at that wavenumber position, and the
  vertical extent of that stripe indicates how broad the peak is — a stripe reaching
  high up the log scale axis is a broad peak, one confined to the lower portion is narrow
  and sharp.  In the consensus spectra, the brightest stripes align precisely with the known Raman marker
  positions, confirming that the consensus averaging process preserves genuine
  biochemical signal rather than introducing artefacts.

- The D6:D5 energy ratio in the consensus spectra (~2.37) establishes a quantitative
  reference for the fat-to-protein/lactose balance of genuine milk. This ratio has direct
  utility as an adulterant detection check: adulteration events that selectively disrupt
  fat content (water addition, vegetable oil substitution) would reduce D6 energy
  disproportionately, while protein or carbohydrate adulteration would shift D5 energy,
  both moving the ratio away from the consensus reference value in a directional and
  biochemically interpretable way.

- Computing the D5 and D6 energy of an unknown test spectrum and comparing against the
  consensus D5/D6 reference values therefore provides a scale-resolved, physically
  grounded complement to the spectral distance function proposed in the broader project
  pipeline — one that not only flags adulteration but offers a first indication of its
  biochemical nature.